In [2]:
!pip install transformers datasets accelerate trl peft bitsandbytes


In [3]:
model_ckpt = "cognitivecomputations/TinyLlama-1.1B-Chat-v1.0"


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch

# Load tokenizer & base model
model_ckpt = "cognitivecomputations/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
tokenizer.pad_token = tokenizer.eos_token

# Load model with 1 output logit
reward_model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=1)

# Prepare dataset
def format_pair(example):
    prompt = example['prompt']
    input_1 = tokenizer(prompt + example['completion_1'], truncation=True, padding='max_length', max_length=512)
    input_2 = tokenizer(prompt + example['completion_2'], truncation=True, padding='max_length', max_length=512)
    return {
        "input_1": input_1,
        "input_2": input_2,
        "ranking": example["ranking"]
    }

dataset = Dataset.from_list(your_dataset)
formatted = dataset.map(format_pair)

# Custom contrastive loss
class RewardTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        input_1 = {k: torch.tensor([x[k] for x in inputs["input_1"]]).to(model.device) for k in inputs["input_1"][0]}
        input_2 = {k: torch.tensor([x[k] for x in inputs["input_2"]]).to(model.device) for k in inputs["input_2"][0]}
        rewards_1 = model(**input_1).logits
        rewards_2 = model(**input_2).logits
        prefs = torch.tensor(inputs["ranking"]).float().unsqueeze(1).to(model.device)
        loss = -torch.nn.functional.logsigmoid((rewards_1 - rewards_2) * (2 * prefs - 1)).mean()
        return loss

# Train the reward model
args = TrainingArguments(output_dir="./tinyllama_reward_model", per_device_train_batch_size=4, num_train_epochs=3)
trainer = RewardTrainer(model=reward_model, args=args, train_dataset=formatted)
trainer.train()

reward_model.save_pretrained("./tinyllama_reward_model")
tokenizer.save_pretrained("./tinyllama_reward_model")


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from trl import PPOConfig, PPOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_ckpt = "TinyLlama/TinyLlama-1.1B-Chat-v0.6"
# Load TinyLLaMA
base_model = AutoModelForCausalLM.from_pretrained(model_ckpt, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
tokenizer.pad_token = tokenizer.eos_token

# Load reward model
from transformers import AutoModelForSequenceClassification
reward_model = AutoModelForSequenceClassification.from_pretrained("./tinyllama_reward_model").eval()

# PPO config
ppo_config = PPOConfig(
    model_name=model_ckpt,
    learning_rate=1e-5,
    batch_size=8,
    forward_batch_size=4,
    mini_batch_size=4,
    log_with=None,
)

ppo_trainer = PPOTrainer(config=ppo_config, model=base_model, tokenizer=tokenizer, reward_model=reward_model)

# Training prompts
prompts = [example['prompt'] for example in your_dataset]

# PPO Training loop
for prompt in prompts:
    tokenized_prompt = tokenizer(prompt, return_tensors="pt").to(base_model.device)
    response_ids = base_model.generate(**tokenized_prompt, max_new_tokens=64)
    response = tokenizer.decode(response_ids[0], skip_special_tokens=True)

    # Compute reward
    reward_input = tokenizer(prompt + response, return_tensors="pt", truncation=True, max_length=512).to(reward_model.device)
    reward = reward_model(**reward_input).logits[0].detach().cpu().item()

    # PPO step
    ppo_trainer.step([prompt], [response], [reward])


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gadep\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v0.6. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [1]:
import json

def load_feedback(path="user_feedback.json"):
    with open(path, "r") as f:
        data = json.load(f)
    return data

def update_dataset(old_dataset, new_data):
    return old_dataset + new_data  # merge lists


In [2]:
import json
from datetime import datetime

def log_feedback(prompt, completions, preferred_index, user_id=None):
    entry = {
        "timestamp": datetime.utcnow().isoformat(),
        "prompt": prompt,
        "completion_1": completions[0],
        "completion_2": completions[1],
        "preferred": preferred_index,
        "user_id": user_id
    }

    with open("feedback_log.jsonl", "a") as f:
        f.write(json.dumps(entry) + "\n")


In [4]:
!pip install gradio


   ---------------------------------------- 0.0/59.5 MB ? eta -:--:--
    --------------------------------------- 1.0/59.5 MB 5.6 MB/s eta 0:00:11
   - -------------------------------------- 2.6/59.5 MB 6.3 MB/s eta 0:00:10
   -- ------------------------------------- 3.7/59.5 MB 6.4 MB/s eta 0:00:09
   --- ------------------------------------ 5.2/59.5 MB 6.4 MB/s eta 0:00:09
   ---- ----------------------------------- 6.6/59.5 MB 6.4 MB/s eta 0:00:09
   ----- ---------------------------------- 7.6/59.5 MB 6.3 MB/s eta 0:00:09
   ------ --------------------------------- 9.2/59.5 MB 6.3 MB/s eta 0:00:08
   ------ --------------------------------- 10.2/59.5 MB 6.3 MB/s eta 0:00:08
   ------- -------------------------------- 11.5/59.5 MB 6.3 MB/s eta 0:00:08
   -------- ------------------------------- 12.8/59.5 MB 6.3 MB/s eta 0:00:08
   --------- ------------------------------ 14.2/59.5 MB 6.3 MB/s eta 0:00:08
   ---------- ----------------------------- 15.7/59.5 MB 6.3 MB/s eta 0:00:07
 

In [1]:
import gradio as gr

def generate_response(prompt):
    # Return two completions
    return "Completion A...", "Completion B..."

def vote_callback(prompt, c1, c2, vote):
    log_feedback(prompt, [c1, c2], vote)

iface = gr.Interface(
    fn=generate_response,
    inputs="text",
    outputs=["text", "text"],
    live=True
)

iface.launch()


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [2]:
import json

def load_feedback_dataset(feedback_log_path="feedback_log.jsonl"):
    data = []
    with open(feedback_log_path, "r") as f:
        for line in f:
            entry = json.loads(line.strip())
            if "preferred" in entry:
                data.append({
                    "prompt": entry["prompt"],
                    "completion_1": entry["completion_1"],
                    "completion_2": entry["completion_2"],
                    "ranking": entry["preferred"]
                })
    return data
